# 2주차 금융 선택과제 · 학생용 빈칸

## 과제의 업무 목표
12개 거래 로그를 합쳐 가맹점 240곳의 거래 요약표를 만들고, 3주차 거래 모니터링 API의 입력 파일을 준비합니다.

1주차에는 기준표 한 개를 정리했습니다. 이번에는 같은 ID가 여러 날짜에 반복되는 로그 7,200행을 다룹니다.

진행 순서: **12개 파일 찾기 → 출처를 남기며 합치기 → 숫자·날짜 정리 → 작은 넓은 표 만들기 → ID별 요약 → 1주차 기준표와 결합 → Pipeline 저장 → 3주차용 CSV 만들기**

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

def find_assignment_dir():
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        candidates = [
            base,
            base / "course_materials" / "assignments" / "week2" / "option2_finance",
            base / "assignments" / "week2" / "option2_finance",
        ]
        for candidate in candidates:
            if (candidate / "data" / "transaction_batches/transaction_batch_01.csv").exists():
                return candidate
    raise FileNotFoundError("transaction_batches/transaction_batch_01.csv를 찾지 못했습니다. assignments 폴더가 포함된 프로젝트를 VS Code에서 열어 주세요.")

ASSIGNMENT_DIR = find_assignment_dir()
DATA_DIR = ASSIGNMENT_DIR / "data"
OUTPUT_DIR = ASSIGNMENT_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
print("과제 폴더:", ASSIGNMENT_DIR)

## Q1. 이름 규칙에 맞는 CSV 12개를 찾습니다

`glob`은 폴더에서 패턴과 맞는 파일 경로를 찾습니다. `sorted`로 순서를 고정한 뒤 개수와 첫·마지막 이름을 확인합니다.

In [ ]:
batch_dir = DATA_DIR / "transaction_batches"
batch_files = sorted(batch_dir.glob("FILL_ME"))
print(len(batch_files), batch_files[0].name, batch_files[-1].name)
assert len(batch_files) == 12

## Q2. 각 파일을 DataFrame으로 읽고 출처 열을 붙인 뒤 하나로 합칩니다

반복 중에는 작은 DataFrame을 `frames` list에 모읍니다. 반복이 끝난 뒤 `concat`으로 한 번에 위아래로 붙입니다.

In [ ]:
frames = []
for file_path in batch_files:
    batch = pd.FILL_ME(file_path)
    batch["source_file"] = file_path.name
    frames.append(batch)

activity = pd.FILL_ME(frames, ignore_index=FILL_ME)
print(activity.shape)
print(activity["source_file"].nunique())
assert len(activity) == 7200
assert activity["source_file"].nunique() == 12

## Q3. 계산할 열과 날짜를 정리하고 4개 ID×7일의 작은 넓은 표를 만듭니다

전체 7,200행을 바로 펼치지 않습니다. 먼저 4개 ID와 7일만 골라 `pivot_table`의 행·열·값이 무엇을 뜻하는지 확인합니다.

In [ ]:
activity["processing_ms"] = pd.FILL_ME(activity["processing_ms"], errors="FILL_ME")
activity["amount_krw"] = pd.to_numeric(activity["amount_krw"], errors="coerce")
activity["transaction_dt"] = pd.FILL_ME(activity["transaction_datetime"], errors="FILL_ME")
activity["activity_date"] = activity["transaction_dt"].dt.date

sample_ids = activity["merchant_id"].drop_duplicates().head(4).tolist()
sample_dates = sorted(activity["activity_date"].dropna().unique())[:7]
small = activity.query("merchant_id in @sample_ids and activity_date in @sample_dates")
wide = small.FILL_ME(index="merchant_id", columns="activity_date", values="amount_krw", aggfunc="sum", fill_value=0)
long_again = wide.reset_index().FILL_ME(id_vars="merchant_id", var_name="activity_date", value_name="amount_krw")
print(wide.shape, long_again.shape)
display(wide)
assert wide.shape == (4, 7)
assert long_again.shape == (28, 3)

## Q4. 여러 로그를 ID 한 개당 한 행으로 요약합니다

이 단계는 7,200행을 버리는 작업이 아닙니다. 같은 대상의 30일 기록을 합계·평균·최근 시각으로 요약해 240행을 만듭니다.

In [ ]:
summary = activity.FILL_ME("merchant_id", as_index=False).agg(
        transaction_count=("transaction_id", "count"),
        active_days=("transaction_dt", lambda s: s.dt.date.nunique()),
        total_amount_krw=("amount_krw", "sum"),
        avg_amount_krw=("amount_krw", "mean"),
        approval_rate=("approved", "mean"),
        refund_rate_30d=("refund_flag", "mean"),
        avg_processing_ms=("processing_ms", "mean"),
        risk_signal_total=("risk_signal_count", "sum"),
        cross_border_count=("country_code", lambda s: (s != "KR").sum()),
        latest_transaction_at=("transaction_dt", "max")
    )
print(summary.shape)
print(summary.head(2))
assert len(summary) == 240
assert summary["merchant_id"].is_unique

## Q5. 1주차 기준표와 2주차 로그 요약표를 ID로 연결합니다

`master`는 대상 240개의 변하지 않거나 천천히 바뀌는 정보입니다. `summary`는 최근 30일 동안 쌓인 활동 정보입니다. 두 표를 같은 ID로 붙여 모델과 API가 읽을 한 행을 만듭니다.

In [ ]:
master = pd.read_csv(DATA_DIR / "merchant_master_reference.csv")
model_table = master.FILL_ME(summary, on="merchant_id", how="FILL_ME", validate="FILL_ME")
print(master.shape, summary.shape, model_table.shape)
assert len(model_table) == 240
assert model_table["merchant_id"].is_unique

## Q6. 숫자와 문자 처리 규칙을 하나의 전처리기로 묶고 저장합니다

전처리기는 열 이름 목록과 처리 순서를 기억합니다. 3주차에는 이 파일을 다시 불러 같은 열 구조를 만들 수 있습니다.

In [ ]:
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_features = ['monthly_sales_10k_krw', 'refund_rate', 'chargeback_rate', 'transaction_count', 'active_days', 'total_amount_krw', 'avg_amount_krw', 'approval_rate', 'refund_rate_30d', 'avg_processing_ms', 'risk_signal_total', 'cross_border_count']
categorical_features = ['industry', 'region', 'kyc_status', 'risk_tier']
input_features = numeric_features + categorical_features
model_input = model_table[input_features].copy()

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features),
])

processed = preprocessor.FILL_ME(model_input)
artifact_dir = ASSIGNMENT_DIR / "artifacts"
artifact_dir.mkdir(exist_ok=True)
artifact_path = artifact_dir / "finance_preprocessor.joblib"
joblib.FILL_ME(preprocessor, artifact_path)
print(processed.shape, artifact_path.exists())
assert processed.shape[0] == 240
assert np.isnan(processed).sum() == 0
assert artifact_path.exists()

## Q7. 3주차가 바로 읽을 요약 CSV와 JSON 미리보기를 만듭니다

CSV는 API·DB·Streamlit의 공통 입력이 됩니다. JSON 미리보기는 FastAPI가 여러 행을 보낼 때 사용할 `list[dict]` 모양입니다.

In [ ]:
summary_path = OUTPUT_DIR / "finance_merchant_summary.csv"
model_table.to_csv(summary_path, index=False, encoding="utf-8-sig")

preview_columns = ["merchant_id", "manual_review_required"] + numeric_features[:3]
json_preview = model_table[preview_columns].head(3).astype(object).where(pd.notna(model_table[preview_columns].head(3)), None).to_dict(orient="records")
print(summary_path, model_table.shape)
print(json_preview)
assert summary_path.exists()
assert len(json_preview) == 3

## 선택 확장: 같은 표를 Parquet으로 저장하고 다시 읽습니다

CSV와 달리 Parquet은 숫자·날짜 자료형을 보존하고 대량 데이터를 빠르게 다시 읽는 데 유리합니다. 저장한 뒤 다시 읽어 shape가 같은지 확인합니다.

In [ ]:
parquet_path = OUTPUT_DIR / "finance_merchant_summary.parquet"
model_table.FILL_ME(parquet_path, index=False)
parquet_back = pd.FILL_ME(parquet_path)
print(parquet_back.shape, parquet_path.exists())
assert parquet_back.shape == model_table.shape

## 선택 확장: FastAPI 실행

같은 폴더의 `api_app_student.py` 빈칸을 채운 뒤 터미널에서 실행합니다.

```bash
python -m uvicorn api_app_student:app --reload --host 127.0.0.1 --port 8010
```

브라우저에서 `/health`, `/items?limit=3`, `/docs`를 확인합니다. 3주차에는 이 CSV를 SQLite에 적재하고 Streamlit 대시보드로 연결할 수 있습니다.